<a href="https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Retrieve HF Token from Colab Secrets or environment
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.getenv('HF_TOKEN')

# Connect DuckDB and authenticate with Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DATASET_URL = "hf://datasets/FlyRank/internship-warehouse"
MID_PANEL_PATH = f"{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet"

print("DuckDB initialized and authenticated successfully.")

DuckDB initialized and authenticated successfully.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [12]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Retrieve HF Token from Colab Secrets or environment
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.getenv('HF_TOKEN')

# Connect DuckDB and authenticate with Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DATASET_URL = "hf://datasets/FlyRank/internship-warehouse"
MID_PANEL_PATH = f"{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet"

print("DuckDB initialized and authenticated successfully.")

# Build Feature Vector Query using SQL Window Functions
query_build_feature_vector = f"""
WITH daily_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position, -- Changed from gsc_position
        gsc_data_available,
        -- Label to predict: Clicks on day t+1
        LEAD(gsc_clicks) OVER (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date) AS target_next_day_clicks,
        -- Intentional Leakage Trap for Section 3 testing: Impressions on day t+1
        LEAD(gsc_impressions) OVER (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date) AS trap_next_day_impressions
    FROM read_parquet('{MID_PANEL_PATH}')
    WHERE gsc_data_available IS TRUE
),
raw_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        -- Feature 1: 7-day rolling average clicks
        AVG(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS clicks_7d_avg,

        -- Feature 2: 7-day rolling average impressions
        AVG(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS impressions_7d_avg,

        -- Feature 3: 7-day rolling average SERP position
        AVG(gsc_avg_position) OVER ( -- Changed from gsc_position
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS position_7d_avg,

        -- Feature 4: 7-day aggregate CTR ratio
        (SUM(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) * 1.0 /
        NULLIF(SUM(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 0)) AS ctr_7d,

        -- Feature 5: Calendar weekend indicator
        CASE WHEN DAYOFWEEK(CAST(report_date AS DATE)) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,

        -- Leakage Trap
        trap_next_day_impressions,

        -- Target
        target_next_day_clicks
    FROM daily_data
)
SELECT *
FROM raw_features
WHERE target_next_day_clicks IS NOT NULL
  AND trap_next_day_impressions IS NOT NULL
LIMIT 100000;
"""

# Load into Pandas DataFrame
df_vector = con.sql(query_build_feature_vector).df()

# Handle missing values explicitly (e.g., zero-fill CTR nulls resulting from 0 impressions)
df_vector['ctr_7d'] = df_vector['ctr_7d'].fillna(0.0)
df_vector = df_vector.fillna(0)

print(f"Feature vector created successfully with shape: {df_vector.shape}")
display(df_vector.head())

DuckDB initialized and authenticated successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector created successfully with shape: (100000, 10)


,client_hash_id,content_hash_id,report_date,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,is_weekend,trap_next_day_impressions,target_next_day_clicks
0,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2026-03-01,0.0,3.0,10.666667,0.0,1,1,0
1,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2026-03-02,0.0,2.0,15.333333,0.0,0,8,0
2,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2026-03-03,0.0,4.0,14.847222,0.0,0,6,0
3,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2026-03-04,0.0,4.5,13.677083,0.0,0,5,0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2026-03-05,0.0,4.6,13.061667,0.0,0,1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes & Availability Audit

| Feature Name | Meaning | Missing Value Handling | Available-When? Justification |
| :--- | :--- | :--- | :--- |
| `clicks_7d_avg` | Average daily search clicks over the past 7 days ($t-6$ to $t$). | Defaulted to `0.0` for new items with truncated history. | **Yes.** Uses strictly historical logs recorded up to cutoff date $t$. |
| `impressions_7d_avg` | Average daily search impressions over the past 7 days ($t-6$ to $t$). | Defaulted to `0.0` for new items. | **Yes.** Recorded in telemetry prior to prediction time $t$. |
| `position_7d_avg` | Average SERP ranking position over the past 7 days ($t-6$ to $t$). | Imputed with default baseline `100.0` (unranked) if null. | **Yes.** Rank history is logged prior to prediction moment $t$. |
| `ctr_7d` | 7-day rolling Click-Through Rate ($\frac{\text{clicks}_{7d}}{\text{impressions}_{7d}}$). | Filled with `0.0` when total impressions equal 0 (prevents division by zero). | **Yes.** Derived strictly from past clicks and impressions up to date $t$. |
| `is_weekend` | Binary indicator (1 if Saturday/Sunday, 0 otherwise) for prediction anchor date $t$. | Categorical encoded as binary integer `0` or `1` (No missing values). | **Yes.** Deterministic calendar property known infinitely in advance. |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [13]:
# --- The Leakage Attack Test ---

# Define honest vs. leaky feature sets
honest_features = ['clicks_7d_avg', 'impressions_7d_avg', 'position_7d_avg', 'ctr_7d', 'is_weekend']
leaky_features = honest_features + ['trap_next_day_impressions']
target = 'target_next_day_clicks'

# Sequential time-ordered train/test split (80% train, 20% test)
split_idx = int(len(df_vector) * 0.8)

X_train_leak, X_test_leak = df_vector[leaky_features].iloc[:split_idx], df_vector[leaky_features].iloc[split_idx:]
X_train_hon, X_test_hon = df_vector[honest_features].iloc[:split_idx], df_vector[honest_features].iloc[split_idx:]
y_train, y_test = df_vector[target].iloc[:split_idx], df_vector[target].iloc[split_idx:]

# 1. Attack Test: Train Model WITH Leaky Feature (trap_next_day_impressions)
model_leaky = RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1)
model_leaky.fit(X_train_leak, y_train)
preds_leak = model_leaky.predict(X_test_leak)

r2_leak = r2_score(y_test, preds_leak)
rmse_leak = np.sqrt(mean_squared_error(y_test, preds_leak))

print("=== LEAKAGE HUNT: ATTACK MODEL (LEAKY FEATURE INCLUDED) ===")
print(f"Leaky R² Score: {r2_leak:.4f}  (Suspiciously high performance from future impression telemetry)")
print(f"Leaky RMSE:     {rmse_leak:.4f}")

# 2. Honest Test: Drop Leaky Feature & Evaluate Honest Baseline
model_honest = RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1)
model_honest.fit(X_train_hon, y_train)
preds_hon = model_honest.predict(X_test_hon)

r2_hon = r2_score(y_test, preds_hon)
rmse_hon = np.sqrt(mean_squared_error(y_test, preds_hon))

print("\n=== LEAKAGE HUNT: HONEST MODEL (LEAKY FEATURE REMOVED) ===")
print(f"Honest R² Score: {r2_hon:.4f}  (True realistic production performance)")
print(f"Honest RMSE:     {rmse_hon:.4f}")

# Feature Importance Check to show the leak dominating
importances = pd.Series(model_leaky.feature_importances_, index=leaky_features).sort_values(ascending=False)
print("\n--- Feature Importances in Leaky Model ---")
print(importances)

=== LEAKAGE HUNT: ATTACK MODEL (LEAKY FEATURE INCLUDED) ===
Leaky R² Score: 0.6086  (Suspiciously high performance from future impression telemetry)
Leaky RMSE:     0.5860

=== LEAKAGE HUNT: HONEST MODEL (LEAKY FEATURE REMOVED) ===
Honest R² Score: 0.5593  (True realistic production performance)
Honest RMSE:     0.6219

--- Feature Importances in Leaky Model ---
clicks_7d_avg                0.740982
trap_next_day_impressions    0.126726
ctr_7d                       0.046680
position_7d_avg              0.042206
impressions_7d_avg           0.038698
is_weekend                   0.004708
dtype: float64


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Refused Fields & Exclusion List

1. `trap_next_day_impressions`: **Future Telemetry.** Contains impression volume on day $t+1$, which does not exist at decision moment $t$ and causes massive label leakage.
2. `gsc_data_available` = `FALSE`: **Unverified Data.** Filtered out because non-available rows represent incomplete reporting syncs or missing logs.
3. `client_hash_id` & `content_hash_id`: **Privacy & Identifier Overfitting.** Entity hashes are identifiers rather than generalizable performance signals; including them directly leads to memorization rather than learning search performance dynamics.
4. `same_day_clicks` (`gsc_clicks` on day $t+1$): **Direct Label Leakage.** Exact target metric measured in the same future window $t+1$.
5. `ga4_conversions_7d`: **Non-Uniform Telemetry.** Omitted across early panel slices where GA4 account integration was incomplete to prevent introducing sparse missingness biases.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.